# BeeSpace 02 - Geo IoT e área potencial de biovigilância

Representação de colmeias georreferenciadas, raio de 3 km e status BeeSpace.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
RAIO_KM = 3
AREA_KM2 = np.pi * RAIO_KM**2
AREA_HA = AREA_KM2 * 100

colmeias = pd.DataFrame({
    "id_colmeia": ["C001", "C002", "C003", "C004"],
    "latitude": [-28.9368, -28.9482, -28.9275, -28.9601],
    "longitude": [-51.5489, -51.5620, -51.5303, -51.5755],
    "temp_colmeia": [34.2, 38.1, 33.8, 36.9],
    "umidade_colmeia": [62, 41, 70, 52],
    "variacao_peso_7d": [1.8, -2.3, 0.9, -0.8],
    "atividade_abelhas": [0.82, 0.25, 0.74, 0.48],
    "status_beespace": ["normal", "alerta", "normal", "atencao"]
})
colmeias["raio_km"] = RAIO_KM
colmeias["area_potencial_ha"] = round(AREA_HA, 0)
colmeias

In [ ]:
try:
    import folium

    centro = [colmeias["latitude"].mean(), colmeias["longitude"].mean()]
    mapa = folium.Map(location=centro, zoom_start=12)
    cores = {"normal": "green", "atencao": "orange", "alerta": "red"}

    for _, row in colmeias.iterrows():
        popup = f'''
        <b>{row["id_colmeia"]}</b><br>
        Status: {row["status_beespace"]}<br>
        Temperatura: {row["temp_colmeia"]} °C<br>
        Umidade: {row["umidade_colmeia"]}%<br>
        Variação de peso 7d: {row["variacao_peso_7d"]} kg<br>
        Área potencial: {row["area_potencial_ha"]:.0f} ha
        '''
        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            popup=popup,
            icon=folium.Icon(color=cores[row["status_beespace"]])
        ).add_to(mapa)
        folium.Circle(
            location=[row["latitude"], row["longitude"]],
            radius=RAIO_KM * 1000,
            color=cores[row["status_beespace"]],
            fill=True,
            fill_opacity=0.08
        ).add_to(mapa)

    mapa_path = OUTPUT_DIR / "mapa_colmeias_beespace.html"
    mapa.save(mapa_path)
    print(f"Mapa salvo em: {mapa_path}")
    mapa
except ImportError:
    print("Folium não está instalado. Instale com: pip install folium")

In [ ]:
colmeias.to_csv(OUTPUT_DIR / "colmeias_geo_iot_beespace.csv", index=False)